In [20]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

## Data Preprocessing

### Data Loading

In [21]:
df=pd.read_csv('titanic.csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Drop Irrelevant columns


In [22]:
df.drop(['PassengerId','Ticket','Name'], axis=1, inplace=True)
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,male,22.0,1,0,7.2500,NaN,S
1,1,1,female,38.0,1,0,71.2833,C85,C
2,1,3,female,26.0,0,0,7.9250,NaN,S
3,1,1,female,35.0,1,0,53.1000,C123,S
4,0,3,male,35.0,0,0,8.0500,NaN,S


### Handling Missing Values


In [23]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64

In [24]:
df['Age']=df['Age'].fillna(df['Age'].median())
print(df['Age'].isnull().sum())

0


In [25]:
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
print(df['Embarked'].isnull().sum())

0


In [26]:
df['Cabin']=df['Cabin'].apply(lambda x: 0 if pd.isnull(x) else 1)
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,male,22.0,1,0,7.2500,0,S
1,1,1,female,38.0,1,0,71.2833,1,C
2,1,3,female,26.0,0,0,7.9250,0,S
3,1,1,female,35.0,1,0,53.1000,1,S
4,0,3,male,35.0,0,0,8.0500,0,S


### Feature Engineering

In [27]:
# add a new column
df['familysize']=1+df['SibSp']+df['Parch']
df['isalone']=0
df['isalone']=np.where(df['familysize']==1,1,0)
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,familysize,isalone
0,0,3,male,22.0,1,0,7.2500,0,S,2,0
1,1,1,female,38.0,1,0,71.2833,1,C,2,0
2,1,3,female,26.0,0,0,7.9250,0,S,1,1
3,1,1,female,35.0,1,0,53.1000,1,S,2,0
4,0,3,male,35.0,0,0,8.0500,0,S,1,1


### Splitting the dataset

In [28]:
# Feature selection
X=df.drop(columns='Survived')
Y=df['Survived']

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

### Preprocessing

In [34]:
categorical_cols=['Sex','Embarked']
num_cols=['Fare','Age']

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
preprocessor=ColumnTransformer([
    ('encode',OneHotEncoder(drop='first'),categorical_cols),
    ('scale',StandardScaler(),num_cols)
]
)

### Pipelines

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
pipelines={
    'logistic':Pipeline([
        ('preprocessing',preprocessor),
        ('model',LogisticRegression())
    ]),
    'knc':Pipeline([
        ('preprocessing',preprocessor),
        ('model',KNeighborsClassifier())
    ]),
    'rfc':Pipeline([
        ('preprocessing',preprocessor),
        ('model',RandomForestClassifier())
    ]),
    'gbc':Pipeline([
         ('preprocessing',preprocessor),
         ('model',GradientBoostingClassifier())
    ]),
    'xgc':Pipeline([
        ('preprocessing',preprocessor),
        ('model',XGBClassifier())
    ])
}

### Hyperparameter grids

In [59]:
hyperparameters={
    'logistic':{
        'model__C':[0.001, 0.01, 0.1, 1, 10],
        'model__penalty':['l1', 'l2', 'elasticnet'],
        'model__max_iter':[500,1000,2000],
        'model__solver': ['saga']
    },
    'knc':{
        'model__n_neighbors':[3, 5, 7, 11],
        'model__metric':['euclidean', 'manhattan'],
        'model__weights': ['uniform', 'distance']
    },
    'rfc':{
        "model__n_estimators":[100, 20, 50],
        'model__max_depth':[None,5,10],
        'model__max_features':['sqrt', 'log2', None]
    },
    'gbc':{
        'model__n_estimators':[100, 20, 50],
        'model__learning_rate':[0.01, 0.05, 0.1],
        'model__max_depth':	[3, 5, 8]
    },
    'xgc':{
        'model__n_estimators':[100, 500, 10],
        'model__learning_rate':[0.01, 0.05, 0.1],
        'model__max_depth':	[3, 5, 8]
    }
}

### Model training + tuning + cross-validation

In [60]:
from sklearn.model_selection import GridSearchCV
best_models={}
for name,pipeline in pipelines.items():
    print('---------------',name,'----------------')
    grid=GridSearchCV(
        pipeline,
        hyperparameters[name],
        cv=5,
        scoring='accuracy'
    )
    grid.fit(x_train,y_train)
    best_models[name]=grid.best_estimator_
    print("Best CV score:", grid.best_score_)
    print('--------------------------------------------')
    print()

--------------- logistic ----------------


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
75 fits failed out of a total of 225.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
75 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\Lenovo\A

Best CV score: 0.7850684526740863
--------------------------------------------

--------------- knc ----------------
Best CV score: 0.7851078498965822
--------------------------------------------

--------------- rfc ----------------
Best CV score: 0.8076036639416919
--------------------------------------------

--------------- gbc ----------------
Best CV score: 0.8132670146754656
--------------------------------------------

--------------- xgc ----------------
Best CV score: 0.8048163104501131
--------------------------------------------



### Evaluation on Model

In [66]:
from sklearn.metrics import accuracy_score,classification_report
for name,model in best_models.items():
    print('---------------',name,'----------------')
    y_pred=model.predict(x_test)
    print('accuracy',accuracy_score(y_test,y_pred))
    print('classification report',classification_report(y_test,y_pred))
    print('--------------------------------------------')

--------------- logistic ----------------
accuracy 0.776536312849162
classification report               precision    recall  f1-score   support

           0       0.80      0.83      0.81       105
           1       0.74      0.70      0.72        74

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179

--------------------------------------------
--------------- knc ----------------
accuracy 0.7932960893854749
classification report               precision    recall  f1-score   support

           0       0.81      0.84      0.83       105
           1       0.76      0.73      0.74        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.79       179
weighted avg       0.79      0.79      0.79       179

--------------------------------------------
--------------- rfc ----------------
accuracy 0.7877094972067039
classificatio

In [68]:
final_model=best_models['gbc']

In [69]:
import joblib

joblib.dump(final_model, "model.pkl")


['model.pkl']

In [70]:
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,familysize,isalone
0,0,3,male,22.0,1,0,7.2500,0,S,2,0
1,1,1,female,38.0,1,0,71.2833,1,C,2,0
2,1,3,female,26.0,0,0,7.9250,0,S,1,1


In [73]:
df['SibSp'].unique()

array([1, 0, 3, 4, 2, 5, 8])